# SynBioCrow 2.2 Development Runner\nUnified M6 runner: install, readiness, bounded design, Drive resume, and rescue output.\n

In [ ]:
import os,sys,time,subprocess,shutil,json,zipfile\nfrom pathlib import Path\nfrom google.colab import drive, files\nT0=time.time()\ndef status(p,msg): print(f'[SynBioCrow 2.2] {p:5.1f}% | elapsed {(time.time()-T0)/60:5.1f} min | {msg}',flush=True)\nstatus(1,'mount Google Drive')\ndrive.mount('/content/drive')\nrepo=Path('/content/SynBioCrow')\nif repo.exists(): shutil.rmtree(repo)\nstatus(5,'clone 2.2 consolidation branch')\nsubprocess.run(['git','clone','--depth','1','--branch','develop/2.2-engine-consolidation','https://github.com/theiman112860/SynBioCrow.git',str(repo)],check=True)\nstatus(10,'install SynBioCrow core + DORAnet optional backend')\nsubprocess.run([sys.executable,'-m','pip','install','-q','-e',str(repo)+'[doranet]'],check=True)\n

In [ ]:
# Configuration\nTARGET_SMILES='CCO'\nSINK_SMILES=['CC']\nBACKENDS=['doranet']  # add configured backends such as 'retrobiocat2' when available\nMAX_ROUTE_STEPS=8\nMAX_ROUTES=100\nSTATE_ROOT=Path('/content/drive/MyDrive/SynBioCrow/2_2_runs')\nSTATE_ROOT.mkdir(parents=True,exist_ok=True)\n

In [ ]:
status(18,'backend readiness')\nfrom synbiocrow import SynBioCrowEngine, DesignRequest, design\nengine=SynBioCrowEngine()\nprint(json.dumps(engine.backend_readiness(),indent=2,sort_keys=True))\ndef progress(p,msg): status(20+0.75*p,msg)\nrequest=DesignRequest(target_smiles=TARGET_SMILES,backend_ids=tuple(BACKENDS),sink_smiles=tuple(SINK_SMILES),max_route_steps=MAX_ROUTE_STEPS,max_routes=MAX_ROUTES)\nresult=design(request,engine=engine,state_root=str(STATE_ROOT),resume=True,progress=progress)\nprint('[RUN ID]',result.run_id)\nprint('[CANDIDATES]',len(result.candidates),'[ROUTES]',len(result.routes))\n

In [ ]:
status(96,'persist user-facing result and package rescue ZIP')\nfrom synbiocrow.execution import json_safe\nrun_dir=STATE_ROOT/result.run_id\nuser_json=run_dir/'DESIGN_RESULT.json'\nuser_json.write_text(json.dumps(json_safe(result),indent=2,sort_keys=True)+'\\n')\nrescue=Path('/content')/f'SynBioCrow_{result.run_id}_ALL_OUTPUTS.zip'\nwith zipfile.ZipFile(rescue,'w',zipfile.ZIP_DEFLATED) as z:\n    for p in sorted(run_dir.rglob('*')):\n        if p.is_file(): z.write(p,p.relative_to(run_dir.parent))\npersisted=STATE_ROOT.parent/rescue.name\nshutil.copy2(rescue,persisted)\nstatus(100,f'complete | persisted {persisted}')\nfiles.download(str(rescue))\n

## Rescue download\nRun this only if the automatic browser download did not start.

In [ ]:
from google.colab import files\nfrom pathlib import Path\nhits=sorted(Path('/content/drive/MyDrive/SynBioCrow').glob('SynBioCrow_run-*_ALL_OUTPUTS.zip'),key=lambda p:p.stat().st_mtime,reverse=True)\nassert hits,'No persisted SynBioCrow 2.2 rescue ZIP found'\nprint('[RESCUE]',hits[0])\nfiles.download(str(hits[0]))\n